# Lab 2 - Hybrid RAG (dense embeddings + keyword search)

**Dataset:** 3 **real** IoT component datasheets, downloaded live from the vendors:

| Chip | What it is | Datasheet |
|------|------------|-----------|
| ESP32 | Wi-Fi/Bluetooth microcontroller (pin tables, electrical specs) | espressif.com |
| MPU6050 | 6-axis accelerometer + gyroscope | Adafruit |
| PMS5003 | Laser particulate-matter (air quality) sensor | Plantower |

These are long, dense, number-heavy PDFs - exactly the documents students hit when
building an IoT project. Retrieval is *mandatory*; you cannot paste these into a prompt.

**The problem this lab shows and fixes:**
Pure embedding search (Lab 1's method) is tuned for *meaning*. It is surprisingly bad at
**exact lookups** - a specific timing value, a pin name, a register address. A table full
of numbers does not "sound like" a question the way a descriptive paragraph does.

**The fix: hybrid search** - run embedding search **and** classic keyword search (BM25)
together, and merge the results. This is a standard production RAG technique, not a hack.

```
                +--> dense retriever (FAISS + embeddings)  --+
question  ------|                                            |--> merge (EnsembleRetriever) --> LLM
                +--> keyword retriever (BM25)  --------------+
```


## Step 0 - Install

In [1]:
%pip install -q \
    langchain langchain-community langchain-openai langchain-huggingface \
    langchain-text-splitters faiss-cpu sentence-transformers rank_bm25 \
    pypdf requests gradio

## Step 1 - OpenRouter API key

Free key: https://openrouter.ai/keys . On Colab, add it as a secret named
`OPENROUTER_API_KEY` (key icon, left sidebar). The cell falls back to a hidden prompt.

In [2]:
import os

def load_key(name: str) -> str:
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: loaded from Colab secret"); return v
    except Exception:
        pass
    if os.getenv(name):
        print(f"{name}: loaded from environment"); return os.environ[name]
    from getpass import getpass
    return getpass(f"Paste your {name}: ")

OPENROUTER_API_KEY = load_key("OPENROUTER_API_KEY")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("Key ends with:", OPENROUTER_API_KEY[-4:])

Paste your OPENROUTER_API_KEY: ··········
Key ends with: 23ce


## Step 2 - Pick a working free model on OpenRouter

Free models come and go and sometimes rate-limit. Instead of hard-coding one, we ask
OpenRouter for its current free model list and keep the first few that actually answer a
test prompt.

In [3]:
import requests
from openai import OpenAI

or_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

PREFERRED = [
    "nvidia/nemotron-3-super-120b-a12b:free",
    "minimax/minimax-m2.7:free",
    "z-ai/glm-5.2:free",
]

def discover_free_models():
    try:
        data = requests.get("https://openrouter.ai/api/v1/models", timeout=20).json()["data"]
        live = [m["id"] for m in data if str(m.get("pricing", {}).get("prompt")) == "0"]
    except Exception:
        live = []
    # preferred first, then anything else free
    ordered = [m for m in PREFERRED if m in live] + [m for m in live if m not in PREFERRED]
    return ordered or PREFERRED

def first_working_model(candidates):
    for m in candidates:
        try:
            r = or_client.chat.completions.create(
                model=m, messages=[{"role": "user", "content": "Reply with: OK"}], timeout=30
            )
            if r.choices[0].message.content:
                print("Using model:", m)
                return m
        except Exception as e:
            print("  skip", m, "-", str(e)[:80])
    raise RuntimeError("No free OpenRouter model responded. Try again in a minute.")

TEXT_MODEL = first_working_model(discover_free_models())

  skip nvidia/nemotron-3-super-120b-a12b:free - 'NoneType' object is not subscriptable
Using model: minimax/minimax-m2.7:free


## Step 3 - Download the 3 datasheets and turn each page into a chunk

We keep it simple: **one page = one chunk**. Each chunk records which datasheet and which
page it came from, so every answer is traceable.

In [14]:
import requests, tempfile, os, shutil
from langchain_community.document_loaders import PyPDFLoader

DATASHEETS = {
    "ESP32":    "https://www.espressif.com/sites/default/files/documentation/esp32_datasheet_en.pdf",
    "MPU6050":  "https://cdn-learn.adafruit.com/downloads/pdf/mpu6050-6-dof-accelerometer-and-gyro.pdf",
    "PMS5003":  "https://cdn-shop.adafruit.com/product-files/3686/plantower-pms5003-manual_v2-3.pdf",
    "contacts": "/content/sample_contacts.pdf",   # local path, uploaded via Colab's Files panel
}

all_chunks = []
for name, source in DATASHEETS.items():
    tmp = os.path.join(tempfile.gettempdir(), f"{name}.pdf")
    if source.startswith("http://") or source.startswith("https://"):
        resp = requests.get(source, timeout=60)
        resp.raise_for_status()
        with open(tmp, "wb") as f:
            f.write(resp.content)
    else:
        shutil.copy(source, tmp)   # local file - just copy it into place

    pages = PyPDFLoader(tmp).load()
    for p in pages:
        p.page_content = " ".join(p.page_content.split())
        p.metadata = {"chip": name, "page": p.metadata.get("page", 0) + 1}
        if len(p.page_content) > 50:
            all_chunks.append(p)
    print(f"{name:8s} -> {len(pages)} pages")

total_chars = sum(len(c.page_content) for c in all_chunks)
print(f"\n{len(all_chunks)} page-chunks, {total_chars:,} characters total")
print("No way this fits in one prompt - retrieval is mandatory here.")

ESP32    -> 78 pages
MPU6050  -> 16 pages
PMS5003  -> 15 pages
contacts -> 1 pages

109 page-chunks, 151,703 characters total
No way this fits in one prompt - retrieval is mandatory here.


## Step 4 - Build the DENSE retriever (Lab 1's method)

Embeddings + FAISS, exactly like Lab 1.

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(all_chunks, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("Dense (FAISS) retriever ready -", vectorstore.index.ntotal, "chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dense (FAISS) retriever ready - 109 chunks


## Step 5 - The failure case

Ask a precise, exact-spec question - the kind an engineer actually asks a datasheet:

> *"What is the total response time of the PMS5003 sensor?"*

The real answer (**"Total Response Time <= 10 seconds"**) is in a spec table on
**PMS5003 page 3**. Watch where the dense retriever ranks that page.

In [16]:
SPEC_Q = "What is the total response time of the PMS5003 sensor?"

print("DENSE retrieval for:", SPEC_Q, "\n")
for rank, (doc, score) in enumerate(vectorstore.similarity_search_with_score(SPEC_Q, k=8), 1):
    flag = "  <-- the answer is on this page" if doc.metadata["chip"] == "PMS5003" and doc.metadata["page"] == 3 else ""
    print(f"  #{rank}  dist={score:.3f}  [{doc.metadata['chip']} p{doc.metadata['page']}]  {doc.page_content[:70]}...{flag}")

DENSE retrieval for: What is the total response time of the PMS5003 sensor? 

  #1  dist=1.080  [PMS5003 p2]  2016 product data manual of PLANTOWER Overview PMS5003 is a ki nd of d...
  #2  dist=1.109  [PMS5003 p1]  2016 product data manual of PLANTOWER Digital universal particle conce...
  #3  dist=1.183  [PMS5003 p13]  2016 product data manual of PLANTOWER Appendix I：PMS5003 transport pro...
  #4  dist=1.282  [PMS5003 p15]  2016 product data manual of PLANTOWER Appendix II：PMS5003 transport pr...
  #5  dist=1.299  [ESP32 p74]  Revision History Cont’d from previous page Date Version Release notes ...
  #6  dist=1.323  [MPU6050 p10]  83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104...
  #7  dist=1.379  [PMS5003 p3]  2016 product data manual of PLANTOWER Technical Index Parameter Index ...  <-- the answer is on this page
  #8  dist=1.392  [ESP32 p45]  4 Functional Description Pin Assignment The 10 capacitive-sensing GPIO...


Typically the PMS5003 spec table lands **low or outside the top 5** - pushed down by the
product-overview paragraphs on pages 1-2, which *talk about* the sensor in fluent prose
and score as more "similar" to the question than a table of raw numbers does.

That is the well-known weakness of embedding search: **great at meaning, weak at exact
technical lookups.**

## Step 6 - The keyword retriever (BM25)

**BM25** is the classic search-engine algorithm - it scores documents by *exact word
overlap*, weighting rare words more. Opposite strengths to embeddings: it does not
understand paraphrase, but it is excellent at exact terms like "response time".

In [17]:
from langchain_community.retrievers import BM25Retriever

# lower-case tokens so a title-case "Total Response Time" in the PDF matches a
# lower-case query - BM25Retriever does NOT lowercase by default
LC = lambda t: t.lower().split()

bm25_retriever = BM25Retriever.from_documents(all_chunks, preprocess_func=LC)
bm25_retriever.k = 5

print("BM25 retrieval for:", SPEC_Q, "\n")
for rank, doc in enumerate(bm25_retriever.invoke(SPEC_Q), 1):
    flag = "  <-- correct page" if doc.metadata["chip"] == "PMS5003" and doc.metadata["page"] == 3 else ""
    print(f"  #{rank}  [{doc.metadata['chip']} p{doc.metadata['page']}]  {doc.page_content[:70]}...{flag}")

BM25 retrieval for: What is the total response time of the PMS5003 sensor? 

  #1  [PMS5003 p3]  2016 product data manual of PLANTOWER Technical Index Parameter Index ...  <-- correct page
  #2  [PMS5003 p1]  2016 product data manual of PLANTOWER Digital universal particle conce...
  #3  [PMS5003 p2]  2016 product data manual of PLANTOWER Overview PMS5003 is a ki nd of d...
  #4  [ESP32 p23]  3 Boot Configurations T able 3-2. Description of Timing Parameters for...
  #5  [ESP32 p19]  2 Pins The internal LDO can be configured as having 1.8 V , or the sam...


BM25 usually puts **PMS5003 page 3 at or near #1** - the exact phrase from the question matches the table's own wording.

## Step 7 - Hybrid = dense + keyword, merged

We fuse the two ranked lists ourselves with **Reciprocal Rank Fusion (RRF)** - the same
method LangChain's `EnsembleRetriever` uses internally, written out so every step is
visible:

- each retriever contributes `weight / (k_const + rank)` for every doc it returns
  (`k_const = 60` is the standard constant; a doc ranked #1 scores more than #2, etc.)
- add the contributions per doc, sort by total

`dense_weight` slides the trust: push toward BM25 (low) for exact-lookup corpora, toward
dense (high) for conceptual questions.

In [18]:
def dense_search(question, k=5):
    return vectorstore.similarity_search(question, k=k)

def bm25_search(question, k=5):
    r = BM25Retriever.from_documents(all_chunks, preprocess_func=LC); r.k = k
    return r.invoke(question)

def hybrid_search(question, dense_weight=0.5, k=5, k_const=60):
    dense_hits = dense_search(question, k=k)
    bm25_hits = bm25_search(question, k=k)

    scores, seen = {}, {}
    for rank, d in enumerate(dense_hits):
        key = (d.metadata["chip"], d.metadata["page"])
        scores[key] = scores.get(key, 0) + dense_weight / (k_const + rank)
        seen[key] = d
    for rank, d in enumerate(bm25_hits):
        key = (d.metadata["chip"], d.metadata["page"])
        scores[key] = scores.get(key, 0) + (1 - dense_weight) / (k_const + rank)
        seen.setdefault(key, d)

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [seen[key] for key in ranked[:k]]

print("HYBRID retrieval for:", SPEC_Q, "\n")
for rank, doc in enumerate(hybrid_search(SPEC_Q, dense_weight=0.5), 1):
    flag = "  <-- correct page" if doc.metadata["chip"] == "PMS5003" and doc.metadata["page"] == 3 else ""
    print(f"  #{rank}  [{doc.metadata['chip']} p{doc.metadata['page']}]  {doc.page_content[:70]}...{flag}")

HYBRID retrieval for: What is the total response time of the PMS5003 sensor? 

  #1  [PMS5003 p2]  2016 product data manual of PLANTOWER Overview PMS5003 is a ki nd of d...
  #2  [PMS5003 p1]  2016 product data manual of PLANTOWER Digital universal particle conce...
  #3  [PMS5003 p3]  2016 product data manual of PLANTOWER Technical Index Parameter Index ...  <-- correct page
  #4  [PMS5003 p13]  2016 product data manual of PLANTOWER Appendix I：PMS5003 transport pro...
  #5  [PMS5003 p15]  2016 product data manual of PLANTOWER Appendix II：PMS5003 transport pr...


## Step 8 - Generate answers and compare

Same LLM, same question, only the retriever differs.

In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model=TEXT_MODEL,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0,
)

import time
def invoke_with_retry(chain, payload, tries=4):
    """Free OpenRouter endpoints occasionally return 429/502/503 - retry with backoff."""
    err = None
    for i in range(tries):
        try:
            return chain.invoke(payload)
        except Exception as e:
            err = e
            print(f"   (LLM retry {i + 1}/{tries}: {str(e)[:90]})")
            time.sleep(2 * (i + 1))
    raise err

ANSWER_PROMPT = ChatPromptTemplate.from_template(
    """Answer the QUESTION using ONLY the CONTEXT. Name the datasheet and page you used.
If the context does not contain the answer, say so.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
)

def fmt(docs):
    return "\n\n".join(f"[{d.metadata['chip']} page {d.metadata['page']}] {d.page_content}" for d in docs)

def answer_with(search_fn, question, k=4):
    docs = search_fn(question)[:k]
    context = fmt(docs)
    ans = invoke_with_retry(ANSWER_PROMPT | llm | StrOutputParser(),
                            {"context": context, "question": question})
    return ans, docs

print("QUESTION:", SPEC_Q)
print("\n--- DENSE-ONLY answer ---")
a1, d1 = answer_with(lambda q: dense_search(q, k=5), SPEC_Q)
print("pages used:", [(d.metadata['chip'], d.metadata['page']) for d in d1])
print(a1)
print("\n--- HYBRID answer ---")
a2, d2 = answer_with(lambda q: hybrid_search(q, dense_weight=0.5), SPEC_Q)
print("pages used:", [(d.metadata['chip'], d.metadata['page']) for d in d2])
print(a2)

QUESTION: What is the total response time of the PMS5003 sensor?

--- DENSE-ONLY answer ---
pages used: [('PMS5003', 2), ('PMS5003', 1), ('PMS5003', 13), ('PMS5003', 15)]

The provided context does not include any information about the total response time of the PMS5003 sensor. Therefore, I cannot determine the response time from the given material. (No datasheet page was used because the answer is not present in the context.)

--- HYBRID answer ---
pages used: [('PMS5003', 2), ('PMS5003', 1), ('PMS5003', 3), ('PMS5003', 13)]
The sensor’s total response time is ≤10 seconds.  
*Source: PMS5003 data sheet (PLANTOWER 2016 product data manual), page 3.*


### A second exact-lookup example

In [20]:
SPEC_Q2 = "What is the storage temperature range of the PMS5003 sensor?"
print("QUESTION:", SPEC_Q2)
print("\n--- DENSE-ONLY ---");  print(answer_with(lambda q: dense_search(q, k=5), SPEC_Q2)[0])
print("\n--- HYBRID ---");      print(answer_with(lambda q: hybrid_search(q, dense_weight=0.5), SPEC_Q2)[0])

QUESTION: What is the storage temperature range of the PMS5003 sensor?

--- DENSE-ONLY ---
Storage temperature range: **‑40 °C to +80 °C** (PMS5003 data manual, page 3).

--- HYBRID ---
The PMS5003 sensor’s storage temperature range is **-40 °C to +80 °C**【PMS5003 page 3】.


## Step 9 - Gradio app

Type a question, drag the slider between **pure keyword (0.0)** and **pure embeddings
(1.0)**, and see all three retrievers' picks plus the final answer side by side.

In [22]:
import gradio as gr

def hybrid_ui(question, dense_weight):
    if not question.strip():
        return "", "", "", ""
    dense_hits = dense_search(question, k=5)
    bm25_hits = bm25_search(question, k=5)
    ans, hyb_hits = answer_with(lambda q: hybrid_search(q, dense_weight=dense_weight), question, k=4)

    def show(docs):
        return "\n".join(f"#{i}  [{d.metadata['chip']} p{d.metadata['page']}]  {d.page_content[:120]}..."
                          for i, d in enumerate(docs, 1))
    return show(dense_hits), show(bm25_hits), show(hyb_hits), ans

with gr.Blocks(title="Lab 2 - Hybrid RAG") as demo:
    gr.Markdown("# Lab 2 - Hybrid RAG on IoT datasheets\nCompare embedding search, keyword search, and the two merged.")
    q = gr.Textbox(label="Question", value="What is the total response time of the PMS5003 sensor?")
    w = gr.Slider(0.0, 1.0, value=0.5, step=0.1, label="Retriever weight  (0 = only keyword/BM25,  1 = only embeddings)")
    btn = gr.Button("Search + Answer", variant="primary")
    ans = gr.Textbox(label="Final answer (hybrid)", lines=4)
    with gr.Row():
        d_box = gr.Textbox(label="Dense (embeddings) top 5", lines=10)
        b_box = gr.Textbox(label="BM25 (keyword) top 5", lines=10)
        h_box = gr.Textbox(label="Hybrid (merged) top 5", lines=10)
    gr.Examples([
        ["What is the total response time of the PMS5003 sensor?", 0.5],
        ["What is the storage temperature range of the PMS5003 sensor?", 0.3],
        ["What is the maximum consistency error of the PMS5003 for PM2.5?", 0.3],
        ["How does the particulate sensor work in principle?", 0.9],
    ], inputs=[q, w])
    btn.click(hybrid_ui, [q, w], [d_box, b_box, h_box, ans])

demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cbe5db4ff2a82d668c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Recap

- Embedding search is weak on **exact** lookups (values, pin names, part numbers).
  We reproduced that failure on a real datasheet, not just described it.
- **BM25** keyword search has the opposite strengths.
- **`EnsembleRetriever`** merges both. One line, standard LangChain, and it recovers the
  page pure embedding search kept missing.

### Exercises
1. Set the slider to 1.0 (pure embeddings) and re-ask the response-time question. Does the
   answer degrade? Now 0.0 (pure BM25) - ask a *conceptual* question ("how does the sensor
   work?"). Which setting wins for which question type?
2. Add a 4th datasheet URL to `DATASHEETS` and re-run - no other code changes needed.
3. In Step 7, change `weights` to `[0.2, 0.8]` and see how the hybrid ranking shifts.
